# Teste do ChatLLMOrchestrator

Este notebook testa as principais funcionalidades do módulo `chat_llm_orchestrator.py`

In [1]:
import sys
import os
sys.path.append('..')

from src.settings import DEFAULT_LLM_MODEL, DEFAULT_TEMPERATURE
import os

[DEBUG] 6969.4482s - Iniciando settings.py
[DEBUG] Carregado SETTINGS em 0.0034s


## 1. Configuração Inicial

Primeiro, vamos configurar as variáveis necessárias para os testes

In [ ]:
# Configurações para teste
API_KEY = '...'
MODEL_NAME = "gpt-5-nano"
TEMPERATURE = 0.3

# Exemplo de contexto do documento
DOCUMENT_CONTEXT = """
Este é um documento de teste que contém informações sobre inteligência artificial.
A IA tem revolucionado diversos setores da indústria e da sociedade.
Modelos de linguagem como GPT têm demonstrado capacidades impressionantes.
"""

# Instruções do sistema
INSTRUCTIONS = "Você é um assistente especializado em análise de documentos. Responda de forma clara e objetiva."

## 2. Teste de Inicialização do Cliente

In [3]:
from src.core.chat_llm_orchestrator import ChatLLMOrchestrator

orchestrator = ChatLLMOrchestrator()
orchestrator._initialize_client(API_KEY)
print("Cliente inicializado com sucesso!" if orchestrator.client else "Falha na inicialização do cliente")

[DEBUG] 6978.0980s - Iniciando utils.py


[DEBUG] Carregado UTILS em 0.1053s

Cliente inicializado com sucesso!


## 3. Teste de Construção dos Itens de Entrada

In [4]:
# História de exemplo
history = [
    {"role": "user", "content": "Qual é o tema principal do documento?"},
    {"role": "assistant", "content": "O tema principal é inteligência artificial."}
]

# Nova pergunta
user_question = "Que setores são mencionados no documento?"

# Construir itens de entrada
input_items = orchestrator._build_input_items(DOCUMENT_CONTEXT, history, user_question)

print("\nItens de entrada construídos:")
for item in input_items:
    print(f"\nRole: {item['role']}")
    print(f"Content: {item['content'][:100]}...")


Itens de entrada construídos:

Role: user
Content: Considere o conteúdo transcrito abaixo como contexto para as perguntas que farei a seguir:

--- INÍC...

Role: assistant
Content: Entendido. Estou pronto para responder perguntas sobre o documento fornecido....

Role: user
Content: Qual é o tema principal do documento?...

Role: assistant
Content: O tema principal é inteligência artificial....

Role: user
Content: Que setores são mencionados no documento?...


## 4. Teste de Geração de Resposta

In [15]:
# Configuração dos provedores LLM para teste
loaded_llm_providers = [{
    "provider": "openai",
    "models": [{
        "name": MODEL_NAME,
        "input_price": 0.01,
        "output_price": 0.03
    }]
}]

# Gerar resposta
response_generator = orchestrator.generate_response(
    api_key=API_KEY,
    model_name=MODEL_NAME,
    instructions=INSTRUCTIONS,
    document_context=DOCUMENT_CONTEXT,
    history=history,
    user_question=user_question,
    loaded_llm_providers=loaded_llm_providers,
    temperature=TEMPERATURE,
    #reasoning_mode="high",
    #verbosity_level="high"
)

# Processar a resposta
full_response = ""
for response in response_generator:
    if response["type"] == "chunk":
        full_response += response["content"]
        print(response["content"], end="")
    elif response["type"] == "final_metrics":
        print("\n\nMétricas finais:")
        print(response["data"])
    elif response["type"] == "error":
        print(f"\nErro: {response['content']}")

Configuração do provedor 'openai' não encontrada para cálculo de custo.


Indústria e sociedade.

Métricas finais:
{'input_tokens': 170, 'cached_tokens': 0, 'output_tokens': 203, 'total_tokens': 373, 'total_cost_usd': 0.0}


## 5. Análise dos Resultados

In [ ]:
print(f"Resposta completa recebida: {len(full_response)} caracteres")
print("\nTeste concluído!")